# 02 — Clean & Feature Engineering

**Purpose:** Load raw flight data, handle missing values, engineer cost and time features, validate the result, and save a clean Parquet file ready for analysis.

**Input:**
- `data/flights_2024.parquet` — raw combined flight data from `01_extract.ipynb`

**Output:**
- `data/flights_clean.parquet` — cleaned and feature-enriched flight data

**Engineered features:**

| Column | Description |
|---|---|
| `DEP_DELAY_COST` | Estimated cost of departure delay: `DEP_DELAY × $100.76`, clipped at 0 |
| `CARRIER_COST` | Cost attributable to carrier delay |
| `WEATHER_COST` | Cost attributable to weather delay |
| `NAS_COST` | Cost attributable to NAS (air traffic control) delay |
| `SECURITY_COST` | Cost attributable to security delay |
| `LATE_AIRCRAFT_COST` | Cost attributable to late incoming aircraft |
| `TIME_BLOCK` | Departure time group: `Morning` (05:00–11:59), `Afternoon` (12:00–17:59), `Night` (18:00–04:59) |
| `IS_WEEKEND` | `True` if flight departs on Saturday or Sunday |

> Cost rate: $100.76 per block minute — [airlines.org](https://www.airlines.org/dataset/u-s-passenger-carrier-delay-costs/)

## Imports

In [12]:
# Add imports here
import pandas as pd
import logging
from pathlib import Path

import sys
sys.path.append("..")   # or "." if running from data/ folder
from utils.logger import get_logger

## Load Data

In [13]:
logging = get_logger()                                                                                                                                                                                                   
   

In [14]:
# Load data/flights_2024.parquet
df = pd.read_parquet("data/flights_2024.parquet")
logging.info(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

2026-04-09 17:10:08,542 — Loaded: 7,079,081 rows, 21 columns


## Handle Missing Values

In [15]:
df.isna().sum()

YEAR                         0
MONTH                        0
DAY_OF_WEEK                  0
OP_UNIQUE_CARRIER            0
ORIGIN                       0
ORIGIN_CITY_NAME             0
DEST                         0
DEST_CITY_NAME               0
CRS_DEP_TIME                 0
DEP_DELAY                92970
DEP_DEL15                92970
DISTANCE                     0
CARRIER_DELAY          5629111
WEATHER_DELAY          5629111
NAS_DELAY              5629111
SECURITY_DELAY         5629111
LATE_AIRCRAFT_DELAY    5629111
source_file                  0
MONTH_NAME                   0
DAY                          0
AIRLINE                      0
dtype: int64

In [16]:
# Drop rows missing DEP_DELAY or DEP_DEL15
before_drop = len(df)
df = df.dropna(subset=['DEP_DELAY', 'DEP_DEL15']).copy()
logging.info(f"Dropped {before_drop - len(df):,} rows missing DEP_DELAY/DEP_15")
# Fill delay cause columns (CARRIER_DELAY, WEATHER_DELAY, etc.) with 0
delay_cols = [    
    'CARRIER_DELAY',
    'WEATHER_DELAY',
    'NAS_DELAY',
    'SECURITY_DELAY',
    'LATE_AIRCRAFT_DELAY'
]

df[delay_cols] = df[delay_cols].fillna(0)
df.isna().sum()

2026-04-09 17:10:10,633 — Dropped 92,970 rows missing DEP_DELAY/DEP_15


YEAR                   0
MONTH                  0
DAY_OF_WEEK            0
OP_UNIQUE_CARRIER      0
ORIGIN                 0
ORIGIN_CITY_NAME       0
DEST                   0
DEST_CITY_NAME         0
CRS_DEP_TIME           0
DEP_DELAY              0
DEP_DEL15              0
DISTANCE               0
CARRIER_DELAY          0
WEATHER_DELAY          0
NAS_DELAY              0
SECURITY_DELAY         0
LATE_AIRCRAFT_DELAY    0
source_file            0
MONTH_NAME             0
DAY                    0
AIRLINE                0
dtype: int64

## Feature Engineering

In [17]:
# DELAY_COST: DEP_DELAY * 100.76  (source: https://www.airlines.org/dataset/u-s-passenger-carrier-delay-costs/)
# Per-cause cost columns: CARRIER_COST, WEATHER_COST, NAS_COST, SECURITY_COST, LATE_AIRCRAFT_COST
COST_PER_MINUTE = 100.76
df['DEP_DELAY_COST'] = (df['DEP_DELAY'].clip(lower=0)) * COST_PER_MINUTE # NEGATIVE IS EARLY DEPARTURE
df['CARRIER_COST'] = df['CARRIER_DELAY'] * COST_PER_MINUTE
df['WEATHER_COST'] = df['WEATHER_DELAY'] * COST_PER_MINUTE
df['NAS_COST'] = df['NAS_DELAY'] * COST_PER_MINUTE
df['SECURITY_COST'] = df['SECURITY_DELAY'] * COST_PER_MINUTE
df['LATE_AIRCRAFT_COST'] = df['LATE_AIRCRAFT_DELAY'] * COST_PER_MINUTE

In [18]:
# TIME_BLOCK: bin CRS_DEP_TIME into Morning / Afternoon / Night
def get_time_block(time):
    hour = int(time//100)

    if 5 <= hour < 12:      return "Morning"
    elif 12 <= hour <= 17:  return "Afternoon"
    else:                   return "Night"
    

df['TIME_BLOCK'] = df['CRS_DEP_TIME'].apply(get_time_block)


In [19]:
# IS_WEEKEND: True if DAY_NAME is Saturday or Sunday
df['IS_WEEKEND'] = df['DAY'].isin(['Saturday', 'Sunday'])

In [20]:
print(df[["DEP_DELAY", "DEP_DELAY_COST", "IS_WEEKEND", "TIME_BLOCK"]].describe())
print(df["TIME_BLOCK"].value_counts())
print(df["IS_WEEKEND"].value_counts())

          DEP_DELAY  DEP_DELAY_COST
count  6.986111e+06    6.986111e+06
mean   1.267708e+01    1.609642e+03
std    5.605997e+01    5.539875e+03
min   -9.600000e+01    0.000000e+00
25%   -6.000000e+00    0.000000e+00
50%   -2.000000e+00    0.000000e+00
75%    9.000000e+00    9.068400e+02
max    3.777000e+03    3.805705e+05
TIME_BLOCK
Morning      2915829
Afternoon    2478192
Night        1592090
Name: count, dtype: int64
IS_WEEKEND
False    5052498
True     1933613
Name: count, dtype: int64


## Data Validation

In [21]:
key_cols = ["DEP_DELAY", "DEP_DEL15", "DEP_DELAY_COST", "TIME_BLOCK", "IS_WEEKEND",                                                                                                                                          
              "MONTH_NAME", "DAY", "AIRLINE"]                                                                                                                                                                              
                                                                                                                                                                                                                           
null_counts = df[key_cols].isnull().sum()                                                                                                                                                                                
assert null_counts.sum() == 0, f"Nulls found:\n{null_counts[null_counts > 0]}"                                                                                                                                           
logging.info("No nulls in key columns") 

valid_blocks = {"Morning", "Afternoon", "Night"}                                                                                                                                                                         
unexpected = set(df["TIME_BLOCK"].unique()) - valid_blocks                                                                                                                                                               
assert not unexpected, f"Unexpected TIME_BLOCK values: {unexpected}"

logging.info(f"Final row count: {len(df):,}")
logging.info(f"Dtypes:\n{df.dtypes}")                                                                                                                                                                                    
                  

2026-04-09 17:10:13,110 — No nulls in key columns
2026-04-09 17:10:13,219 — Final row count: 6,986,111
2026-04-09 17:10:13,220 — Dtypes:
YEAR                     int64
MONTH                    int64
DAY_OF_WEEK              int64
OP_UNIQUE_CARRIER       object
ORIGIN                  object
ORIGIN_CITY_NAME        object
DEST                    object
DEST_CITY_NAME          object
CRS_DEP_TIME             int64
DEP_DELAY              float64
DEP_DEL15              float64
DISTANCE               float64
CARRIER_DELAY          float64
WEATHER_DELAY          float64
NAS_DELAY              float64
SECURITY_DELAY         float64
LATE_AIRCRAFT_DELAY    float64
source_file             object
MONTH_NAME              object
DAY                     object
AIRLINE                 object
DEP_DELAY_COST         float64
CARRIER_COST           float64
WEATHER_COST           float64
NAS_COST               float64
SECURITY_COST          float64
LATE_AIRCRAFT_COST     float64
TIME_BLOCK              ob

## Save to Parquet

In [22]:
# Save to data/flights_clean.parquet
output_path = Path("data")/"flights_clean.parquet"

df.to_parquet('data/flights_2024_clean.parquet', index=False)
logging.info(f"Saved {len(df):,} rows to {output_path}")

2026-04-09 17:10:16,065 — Saved 6,986,111 rows to data/flights_clean.parquet
